# GPS trajectory imputation with a Transformer: training and testing of one fold

One run of the notebook = one fold of the cross-validation (`FOLD` = 1–4). The 4 runs are executed in separate folders, with `final_dataset_thess9.csv` in each one.

**Pipeline:** segments → features → stays and trips → (augmentation) → windows → masking → training (selection of E, refit) → prediction on the test gaps.

The settings correspond to the main variant V9. The trainings of the thesis were done without a fixed seed, so a new training gives slightly different values.

## 1. Settings and cross-validation

The 170 days (day change at 05:00) are split into 4 folds of consecutive days. Test = fold `FOLD`. Validation = the last full Monday–Sunday week of each of the other folds. Train = the remaining days of the development set (dev).

In [ ]:
import os, json
import numpy as np, pandas as pd
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

FOLD = int(os.environ.get("FOLD", 1))      # test fold (1..4)
INPUT_CSV = "final_dataset_thess9.csv"
N_FOLDS = 4

# windows
W = 48                                     # window length 
STRIDE = 4                                 # sliding window stride in train
MARGIN = 2                                 # margin around a trip inside the window
MAX_SPAN = W - 2 * MARGIN                  # maximum trip length (steps)
GAP_LENGTHS_EVAL = [3, 6, 11, 20, 31]      # gap lengths in validation and test

# variant V9 
LAMBDA_ORT = 0.5                           # ORT weight
W_STEP = 0.0                               # step loss weight
AUG_TRIPS = False                          # trip augmentation
LR = 1e-3                                  # learning rate
D_MODEL, N_LAYERS, DROPOUT = 128, 3, 0.3   # model size and dropout

DAY_SHIFT = pd.Timedelta(hours=5)          # the day changes at 05:00


def day_str(t):
    """Day 'YYYY-MM-DD' with 05:00 as the boundary."""
    return (pd.to_datetime(t) - DAY_SHIFT).dt.strftime("%Y-%m-%d")


def day_runs(days):
    out = []
    for d in sorted(pd.Timestamp(x) for x in days):
        if out and (d - out[-1][1]).days == 1:
            out[-1][1] = d
        else:
            out.append([d, d])
    return ", ".join(f"{a:%d/%m}" if a == b else f"{a:%d/%m}–{b:%d/%m}" for a, b in out)


def last_full_week(days):
    """Last Monday–Sunday week with data on all 7 days."""
    s = set(days)
    for d in sorted(days, reverse=True):
        wk = [(pd.Timestamp(d) + pd.Timedelta(days=k)).strftime("%Y-%m-%d") for k in range(7)]
        if pd.Timestamp(d).dayofweek == 0 and all(w in s for w in wk):
            return wk
    return []


ALL_DAYS = np.sort(day_str(pd.read_csv(INPUT_CSV, usecols=["real_time"]).real_time).unique())
FOLD_DAYS = {k + 1: list(d) for k, d in enumerate(np.array_split(ALL_DAYS, N_FOLDS))}
VAL_WEEK = {k: last_full_week(d) for k, d in FOLD_DAYS.items()}
TEST_DAYS = FOLD_DAYS[FOLD]
DEV_DAYS = sorted(d for k, v in FOLD_DAYS.items() if k != FOLD for d in v)
VAL_DAYS = sorted(d for k in FOLD_DAYS if k != FOLD for d in VAL_WEEK[k])
TRAIN_DAYS = sorted(set(DEV_DAYS) - set(VAL_DAYS))

print(f"days: {len(ALL_DAYS)}")
for k, d in FOLD_DAYS.items():
    print(f"  fold {k}: {day_runs(d)} ({len(d)}) | validation: {day_runs(VAL_WEEK[k])}")
print(f"FOLD {FOLD}: test {len(TEST_DAYS)} | validation {len(VAL_DAYS)} | train {len(TRAIN_DAYS)} | dev {len(DEV_DAYS)}")

## 2. Segments

New segment after every point with `B ∈ {2, 6}` (recording interruption). `dt_s` = time to the next point of the segment.

In [ ]:
df = pd.read_csv(INPUT_CSV, usecols=["real_time", "x", "y", "B", "dx", "dy"])
df["real_time"] = pd.to_datetime(df.real_time)
df["seg_id"] = df.B.isin([2, 6]).shift(1, fill_value=False).cumsum()
df["dt_s"] = (df.groupby("seg_id").real_time.shift(-1) - df.real_time).dt.total_seconds()
df.to_csv("segments_part1.csv", index=False)
print(f"{df.seg_id.nunique():,} segments | points per segment:\n{df.groupby('seg_id').size().describe().round(1).to_string()}")

## 3. Input features and normalization

- Last point of each segment: `dx = dy = 0`, `dt_s = 60`.
- Step `dx, dy` divided by the p99 of the step length (`STEP_SCALE`), clipped at ±5.
- Hour and day of the week with sine and cosine.
- Position relative to home, divided by `SCALE_M` (p98.7 of the distance from home).

In [ ]:
df = pd.read_csv("segments_part1.csv")
df["real_time"] = pd.to_datetime(df.real_time)

df["is_seg_start"] = (df.groupby("seg_id").cumcount() == 0).astype(int)
df["is_seg_end"] = df.dt_s.isna().astype(int)
df.loc[df.is_seg_end == 1, ["dx", "dy", "dt_s"]] = [0.0, 0.0, 60.0]

df["dt_norm"] = ((df.dt_s - 60.0) / 60.0).clip(-1, 1)
STEP_SCALE = np.hypot(df.dx, df.dy).quantile(0.99)
CLIP_NORM = 5.0
df["dx_norm"] = (df.dx / STEP_SCALE).clip(-CLIP_NORM, CLIP_NORM)
df["dy_norm"] = (df.dy / STEP_SCALE).clip(-CLIP_NORM, CLIP_NORM)

hod = df.real_time.dt.hour + df.real_time.dt.minute / 60.0 + df.real_time.dt.second / 3600.0
dow = df.real_time.dt.dayofweek + hod / 24.0
df["tod_sin"], df["tod_cos"] = np.sin(2 * np.pi * hod / 24.0), np.cos(2 * np.pi * hod / 24.0)
df["dow_sin"], df["dow_cos"] = np.sin(2 * np.pi * dow / 7.0), np.cos(2 * np.pi * dow / 7.0)

gx, gy = (df.x // 100).astype(int), (df.y // 100).astype(int)
cx, cy = df.groupby([gx, gy]).size().idxmax()
home = (gx == cx) & (gy == cy)
x_home, y_home = df.loc[home, "x"].median(), df.loc[home, "y"].median()
df["xc_m"], df["yc_m"] = df.x - x_home, df.y - y_home
df["d_home_m"] = np.hypot(df.xc_m, df.yc_m)
SCALE_M = df.d_home_m.quantile(0.987)
df["xc_norm"], df["yc_norm"] = df.xc_m / SCALE_M, df.yc_m / SCALE_M

df.to_csv("segments_part2.csv", index=False)
norm_config = {"x_home": float(x_home), "y_home": float(y_home), "SCALE_M": float(SCALE_M), "STEP_SCALE": float(STEP_SCALE)}
json.dump(norm_config, open("norm_config.json", "w"), indent=2)
print(norm_config)
step = np.hypot(df.dx, df.dy)
kmh = step / df.dt_s * 3.6
clipped = (df.dx.abs() / STEP_SCALE > CLIP_NORM) | (df.dy.abs() / STEP_SCALE > CLIP_NORM)
print(f"distance from home: < 50 m {(df.d_home_m < 50).mean():.1%} | mean {df.d_home_m.mean():.0f} m | "
      f"std {df.d_home_m.std():.0f} m")
print(f"outside [-1, 1]^2: {(np.maximum(df.xc_norm.abs(), df.yc_norm.abs()) > 1).mean():.2%} | "
      f"max |x_c|, |y_c|: {np.maximum(df.xc_norm.abs(), df.yc_norm.abs()).max():.2f}")
print(f"step: median {step.median():.1f} m | median at > 20 km/h {step[kmh > 20].median():.0f} m | "
      f"p99 {STEP_SCALE:.0f} m | clipped at ±{CLIP_NORM:.0f}: {int(clipped.sum())} points, "
      f"{kmh[clipped].min():.0f}–{kmh[clipped].max():.0f} km/h")

## 4. Stays and trips

- **Stay:** ≥ 20 minutes with all points within a radius of 150 m from their center.
- **Trip:** movement between two consecutive stays of the same segment, with ≥ 5 steps or displacement > 400 m. Trips at the ends of a segment are discarded.
- A trip > `MAX_SPAN` steps is split at short stays (≥ 5 minutes within 50 m).

Output: `trip_spans.json` (seg_id → [(ts, te), ...]).

In [ ]:
STAY_RADIUS_M, MIN_STAY_MIN = 150, 20
SHORT_STAY_RADIUS_M, SHORT_STAY_MIN = 50, 5
MIN_TRIP_LEN, TRIP_DISP_M = 5, 400

df = pd.read_csv("segments_part2.csv")
df["real_time"] = pd.to_datetime(df.real_time)


def greedy_stays(x, y, t, radius, min_min):
    out, i, n = [], 0, len(x)
    while i < n:
        j = i + 1
        while j < n:
            cx_, cy_ = x[i:j + 1].mean(), y[i:j + 1].mean()
            if np.hypot(x[i:j + 1] - cx_, y[i:j + 1] - cy_).max() > radius:
                break
            j += 1
        if (t[j - 1] - t[i]) / np.timedelta64(1, "m") >= min_min:
            out.append((i, j - 1))
            i = j
        else:
            i += 1
    return out


def keep_trip(x, y, ts, te):
    """≥ MIN_TRIP_LEN steps or displacement > TRIP_DISP_M."""
    steps = te - ts
    return steps >= MIN_TRIP_LEN or (steps > 0 and np.hypot(x[te] - x[ts], y[te] - y[ts]) > TRIP_DISP_M)


trips, n_long, n_split, n_initial = [], 0, 0, 0
for seg_id, g in df.groupby("seg_id"):
    x, y, t = g.x.to_numpy(), g.y.to_numpy(), g.real_time.to_numpy()
    stays = greedy_stays(x, y, t, STAY_RADIUS_M, MIN_STAY_MIN)
    for (_, ts), (te, _) in zip(stays[:-1], stays[1:]):       # from the end of one stay to the start of the next
        if te - ts <= 0 or not keep_trip(x, y, ts, te):
            continue
        n_initial += 1
        pieces = [(ts, te, "full" if te - ts >= MIN_TRIP_LEN else "mini")]
        if te - ts > MAX_SPAN:                                 # long trip: split at short stays
            n_long += 1
            ss = greedy_stays(x[ts + 1:te], y[ts + 1:te], t[ts + 1:te], SHORT_STAY_RADIUS_M, SHORT_STAY_MIN)
            if ss:
                n_split += 1
                b = [ts] + [v + ts + 1 for s_ in ss for v in s_] + [te]
                pieces = [(p0, p1, "split") for p0, p1 in zip(b[0::2], b[1::2]) if keep_trip(x, y, p0, p1)]
        trips += [dict(seg_id=int(seg_id), ts=int(p0), te=int(p1), trip_type=tt) for p0, p1, tt in pieces]

tm = pd.DataFrame(trips)
trip_spans = {}
for r in tm.itertuples():
    trip_spans.setdefault(r.seg_id, []).append((r.ts, r.te))
json.dump(trip_spans, open("trip_spans.json", "w"))

print(f"initial trips: {n_initial} | after splitting: {len(tm)} {tm.trip_type.value_counts().to_dict()} | "
      f"> {MAX_SPAN} steps: {n_long}, of which split {n_split} | "
      f"> {MAX_SPAN} after splitting: {int((tm.te - tm.ts > MAX_SPAN).sum())}")

## 5. Trip augmentation (optional)

 Each trip of the train days is matched to the OSM road network (HMM map matching) and gives 5 variants:
- a different speed profile on the same road, with the same endpoints,
- lateral GPS noise (AR(1)),
- a time shift of ±30 minutes.


In [ ]:
import shutil
from collections import Counter

if not AUG_TRIPS:
    if os.path.exists("aug_info.json"):
        os.remove("aug_info.json")
    print("augmentation: no")
else:
    N_VARS = 5                 # variants per trip
    CONTEXT_PTS = 20           # stay points before and after the trip
    TIME_SHIFT_MIN = 30        # time shift ±30'
    DT_JITTER_S = 2
    WARP_SIGMA, WARP_MODES = 0.25, 3
    LAT_RHO, LAT_JITTER_M, E_CLIP_M, ROAD_SNAP_MAX_M = 0.80, 8.0, 25.0, 20.0
    MAX_MED_DIST, DETOUR_MAX_M, LOOP_CUT_M = 60.0, 150.0, 150.0
    WALK_KMH, SMOOTH_WIN = 1.5, 5
    MATCH_BETA, MATCH_RADIUS, MATCH_KMAX, HEAD_W, HEAD_MIN_M = 15.0, 100.0, 20, 8.0, 30.0
    OSM_GRAPHML, OSM_MARGIN_M = "osm_drive_trips.graphml", 1500.0
    rng = np.random.default_rng(7 + FOLD)

    def path_cum(path):
        """Arc length from the start to each vertex."""
        return np.r_[0.0, np.cumsum(np.hypot(np.diff(path[:, 0]), np.diff(path[:, 1])))]

    def project_to_path(xy, path, cum=None):
        """Projection of points onto the line: position s along it and signed lateral distance e."""
        cum = path_cum(path) if cum is None else cum
        a, v = path[:-1], path[1:] - path[:-1]
        L2 = np.where((v ** 2).sum(1) > 0, (v ** 2).sum(1), 1e-12)
        w = xy[:, None, :] - a[None]
        t = np.clip((w * v[None]).sum(-1) / L2[None], 0.0, 1.0)
        proj = a[None] + t[..., None] * v[None]
        d = np.hypot(xy[:, None, 0] - proj[..., 0], xy[:, None, 1] - proj[..., 1])
        j, i = d.argmin(1), np.arange(len(xy))
        cross = v[j, 0] * w[i, j, 1] - v[j, 1] * w[i, j, 0]
        return cum[j] + t[i, j] * np.sqrt(L2[j]), np.sign(cross) * d[i, j]

    def point_at_arc(path, cum, s):
        """Point and unit normal vector at position s of the arc."""
        s = np.clip(s, 0.0, cum[-1])
        j = np.clip(np.searchsorted(cum, s, side="right") - 1, 0, len(path) - 2)
        seg = path[j + 1] - path[j]
        Ls = np.where(np.hypot(seg[:, 0], seg[:, 1]) > 0, np.hypot(seg[:, 0], seg[:, 1]), 1e-12)
        tg = seg / Ls[:, None]
        return path[j] + ((s - cum[j]) / Ls)[:, None] * seg, np.stack([-tg[:, 1], tg[:, 0]], 1)

    def random_warp(u):
        """new speed profile with the same endpoints."""
        g = np.linspace(0.0, 1.0, 1001)
        a = rng.normal(0.0, WARP_SIGMA, WARP_MODES) / np.arange(1, WARP_MODES + 1)
        Wc = np.cumsum(np.exp(sum(a[k] * np.sin((k + 1) * np.pi * g) for k in range(WARP_MODES))))
        return np.interp(np.clip(u, 0.0, 1.0), g, (Wc - Wc[0]) / (Wc[-1] - Wc[0]))

    def ar1(n, sigma, rho=LAT_RHO):
        if n == 0:
            return np.zeros(0)
        e = np.empty(n); e[0] = rng.normal(0.0, sigma)
        si = sigma * np.sqrt(max(1.0 - rho ** 2, 1e-12))
        for i in range(1, n):
            e[i] = rho * e[i - 1] + rng.normal(0.0, si)
        return e

    def cut_loops(R_, s_xy):
        c = path_cum(R_)
        s0_, _ = project_to_path(s_xy, R_, c)
        ch = np.hypot(np.diff(s_xy[:, 0]), np.diff(s_xy[:, 1]))
        drop, ex = np.zeros(len(R_), bool), []
        for i in np.where((ch < LOOP_CUT_M) & (np.diff(s0_) - ch > DETOUR_MAX_M))[0]:
            a_, b_ = s0_[i], s0_[i + 1]
            if ((s0_ > a_) & (s0_ < b_)).any():          # there are points there: not a loop
                continue
            drop |= (c > a_) & (c < b_)
            ex += [a_, b_]
        if not ex:
            return R_
        ex = np.array(ex); pe, _ = point_at_arc(R_, c, ex)
        s_all, P_all = np.r_[c[~drop], ex], np.vstack([R_[~drop], pe])
        return P_all[np.argsort(s_all, kind="stable")]

    if not os.path.exists("segments_part2_base.csv"):
        shutil.copy("segments_part2.csv", "segments_part2_base.csv")
    if not os.path.exists("trip_spans_base.json"):
        shutil.copy("trip_spans.json", "trip_spans_base.json")
    base = pd.read_csv("segments_part2_base.csv")
    base["real_time"] = pd.to_datetime(base.real_time)
    tspans0 = {int(k): [tuple(s) for s in v] for k, v in json.load(open("trip_spans_base.json")).items()}

    first_idx = base.groupby("seg_id").apply(lambda g: g.index.min(), include_groups=False)
    n_of_seg = base.groupby("seg_id").size()
    AUG_SEG_START = int(base.seg_id.max()) + 1
    bday, train_set = day_str(base.real_time).to_numpy(), set(TRAIN_DAYS)

    passes = []
    for sid, sp in tspans0.items():
        n_seg, i0 = int(n_of_seg[sid]), int(first_idx[sid])
        for ts, te in sp:
            if bday[i0 + ts] not in train_set:
                continue
            a = max(0, ts - CONTEXT_PTS); b = min(n_seg, te + 1 + CONTEXT_PTS)
            ctx = base.iloc[i0 + a: i0 + b]
            passes.append(dict(uid=f"{sid}_{ts}", xy=np.c_[ctx.x.to_numpy(), ctx.y.to_numpy()],
                               dt=ctx.dt_s.to_numpy().astype(float), t0=ctx.real_time.iloc[0],
                               ts_loc=int(ts - a), te_loc=int(te - a)))
    print(f"trips: {sum(len(v) for v in tspans0.values())}, on train days: {len(passes)}")

   
    import osmnx as ox, networkx as nx
    from shapely.geometry import LineString, Point
    from shapely.strtree import STRtree

    if os.path.exists(OSM_GRAPHML):
        G = ox.load_graphml(OSM_GRAPHML)
    else:
        from pyproj import Transformer
        allxy = np.vstack([p["xy"] for p in passes])
        lo, la = Transformer.from_crs("epsg:2100", "epsg:4326", always_xy=True).transform(allxy[:, 0], allxy[:, 1])
        dlat = OSM_MARGIN_M / 111_000.0
        dlon = OSM_MARGIN_M / (111_000.0 * np.cos(np.radians(la.mean())))
        G = ox.graph_from_bbox((lo.min() - dlon, la.min() - dlat, lo.max() + dlon, la.max() + dlat),
                               network_type="drive", simplify=True)
        G = ox.project_graph(G, to_crs="EPSG:2100")
        ox.save_graphml(G, OSM_GRAPHML)
    if G.graph.get("crs") not in ("EPSG:2100", "epsg:2100"):
        G = ox.project_graph(G, to_crs="EPSG:2100")

    def _sub(g, a_, b_):
        """Vertices of line g between lengths a_ and b_."""
        if b_ <= a_:
            p_ = g.interpolate(a_); return [(p_.x, p_.y)]
        cs = np.asarray(g.coords, float)
        dd = np.r_[0.0, np.cumsum(np.hypot(np.diff(cs[:, 0]), np.diff(cs[:, 1])))]
        pa, pb = g.interpolate(a_), g.interpolate(b_)
        return [(pa.x, pa.y)] + [tuple(c) for c, q in zip(cs, dd) if a_ < q < b_] + [(pb.x, pb.y)]

    class RoadNet:
        """Edges of the OSM graph with a spatial index and distances on the network."""

        def __init__(self, G):
            self.G, self.edges, self.geoms, self._dij = G, [], [], {}
            for u, v, k, d in G.edges(keys=True, data=True):
                g = d.get("geometry") or LineString([(G.nodes[u]["x"], G.nodes[u]["y"]),
                                                     (G.nodes[v]["x"], G.nodes[v]["y"])])
                self.edges.append((u, v, k, g)); self.geoms.append(g)
            self.tree = STRtree(self.geoms)

        def _from(self, node, cutoff):
            """Dijkstra distances from a node (with cache)."""
            c, tbl = self._dij.get(node, (0.0, None))
            if tbl is None or cutoff > c:
                try:
                    tbl = nx.single_source_dijkstra_path_length(self.G, node, cutoff=cutoff, weight="length")
                except nx.NodeNotFound:
                    tbl = {}
                self._dij[node] = (cutoff, tbl)
            return tbl

        def candidates(self, pt, radius, k_max):
            """Candidate edges near the point: (edge, distance, position on the edge)."""
            P_ = Point(pt); out = []
            for i in np.atleast_1d(self.tree.query(P_.buffer(radius))):
                g = self.geoms[int(i)]; d = g.distance(P_)
                if d <= radius:
                    out.append((int(i), float(d), float(g.project(P_))))
            return sorted(out, key=lambda r: r[1])[:k_max]

        def route_dist(self, i, oi, j, oj, cutoff):
            """Network distance from (edge i, position oi) to (edge j, position oj)."""
            if i == j:
                return abs(oj - oi)
            u1, v1, _, g1 = self.edges[i]; u2 = self.edges[j][0]
            if v1 == u2:
                return g1.length - oi + oj
            d = self._from(v1, cutoff).get(u2)
            return None if d is None else g1.length - oi + d + oj

        def stitch(self, seq):
            """Continuous road line from the sequence of matched edges."""
            pts = []
            for n_, (i, off) in enumerate(seq):
                g = self.geoms[i]
                if n_ + 1 < len(seq):
                    i2, off2 = seq[n_ + 1]
                    if i2 == i:
                        pts += _sub(g, *sorted((off, off2))); continue
                    if self.edges[i][0] == self.edges[i2][1] and self.edges[i][1] == self.edges[i2][0]:
                        pts += _sub(g, *sorted((off, g.length - off2))); continue   # same road, opposite direction
                    pts += _sub(g, off, g.length)
                    v1, u2 = self.edges[i][1], self.edges[i2][0]
                    if v1 != u2:
                        try:
                            nodes = nx.shortest_path(self.G, v1, u2, weight="length")
                        except (nx.NetworkXNoPath, nx.NodeNotFound):
                            nodes = []
                        for a_, b_ in zip(nodes[:-1], nodes[1:]):
                            gg = min(self.G[a_][b_].values(), key=lambda e: e.get("length", 1e18)).get("geometry")
                            pts += (list(gg.coords) if gg is not None else
                                    [(self.G.nodes[a_]["x"], self.G.nodes[a_]["y"]),
                                     (self.G.nodes[b_]["x"], self.G.nodes[b_]["y"])])
                    pts += _sub(self.geoms[i2], 0.0, off2)
                elif not pts:
                    pts += _sub(g, off, min(off + 1.0, g.length))
            out = [pts[0]]
            for p_ in pts[1:]:
                if abs(p_[0] - out[-1][0]) > 1e-6 or abs(p_[1] - out[-1][1]) > 1e-6:
                    out.append(p_)
            return np.asarray(out, float)

    def _edge_dir(net, j, o):
        """Unit vector of edge j (direction u → v) at position o."""
        g = net.geoms[j]
        a, b = g.interpolate(max(o - 5.0, 0.0)), g.interpolate(min(o + 5.0, g.length))
        v = np.array([b.x - a.x, b.y - a.y]); nv = np.hypot(*v)
        return v / nv if nv > 0 else v

    def match_hmm(xy, net, sigma_z=20.0, beta=MATCH_BETA, radius=MATCH_RADIUS, k_max=MATCH_KMAX, maj_win=SMOOTH_WIN):
        cands_all = [net.candidates(xy[i], radius, k_max) for i in range(len(xy))]
        idx_ok = np.array([i for i in range(len(xy)) if cands_all[i]])
        if len(idx_ok) == 0:
            return None, None
        xy_ok, cands = xy[idx_ok], [cands_all[i] for i in idx_ok]
        m = len(cands)
        hd = xy_ok[np.minimum(np.arange(m) + 1, m - 1)] - xy_ok[np.maximum(np.arange(m) - 1, 0)]
        hn = np.hypot(hd[:, 0], hd[:, 1])
        hd = hd / np.where(hn > 0, hn, 1.0)[:, None]

        def em(i, j, d, o):
            e = -0.5 * (d / sigma_z) ** 2
            if hn[i] > HEAD_MIN_M:                     # penalty only for opposite direction
                e -= HEAD_W * max(0.0, -float(_edge_dir(net, j, o) @ hd[i]))
            return e

        NEG = -1e18
        prev, back = np.array([em(0, j, d, o) for j, d, o in cands[0]]), []
        for i in range(1, m):
            straight = float(np.hypot(*(xy_ok[i] - xy_ok[i - 1])))
            cut = max(4.0 * straight, 300.0)
            cur, bp = np.full(len(cands[i]), NEG), np.full(len(cands[i]), -1, int)
            for b_, (jb, db, ob) in enumerate(cands[i]):
                e_b = em(i, jb, db, ob)
                for a_, (ja, da, oa) in enumerate(cands[i - 1]):
                    if prev[a_] <= NEG / 2:
                        continue
                    dr = net.route_dist(ja, oa, jb, ob, cut)
                    sc = prev[a_] + (NEG if dr is None else -abs(dr - straight) / beta) + e_b
                    if sc > cur[b_]:
                        cur[b_], bp[b_] = sc, a_
            if cur.max() <= NEG / 2:
                return None, None
            prev = cur; back.append(bp)

        seq, a_, dist = [None] * m, int(prev.argmax()), [0.0] * m
        for i in range(m - 1, -1, -1):
            j, d, o = cands[i][a_]
            seq[i], dist[i] = (j, o), d
            if i > 0:
                a_ = int(back[i - 1][a_])
        edges, seq2 = [e for e, _ in seq], list(seq)                # majority smoothing
        for i in range(m):
            window = edges[max(0, i - maj_win // 2): min(m, i + maj_win // 2 + 1)]
            vals, counts = np.unique(window, return_counts=True)
            maj = int(vals[counts.argmax()])
            if maj != edges[i] and counts.max() > len(window) / 2:
                hit = next((c for c in cands[i] if c[0] == maj), None)
                if hit is not None:
                    seq2[i] = (hit[0], hit[2])
        return net.stitch(seq2), float(np.median(dist))

    net = RoadNet(G)
    print(f"road network: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

    ref, kinds = {}, Counter()
    for p in passes:
        s_xy = p["xy"][p["ts_loc"]:p["te_loc"] + 1]
        c_dt = p["dt"][p["ts_loc"]:p["te_loc"] + 1]
        dist = np.hypot(np.diff(s_xy[:, 0]), np.diff(s_xy[:, 1]))
        kmh = path_cum(s_xy)[-1] / max(c_dt[:-1][dist > 0.5].sum(), 1.0) * 3.6
        R_, kind = None, "slow"
        if kmh >= WALK_KMH:
            try:
                cand, med = match_hmm(s_xy, net, maj_win=1)
            except Exception:
                cand, med = None, None
            kind = "no_match"
            if cand is not None and med is not None and med <= MAX_MED_DIST:
                R_ = cut_loops(cand, s_xy)
                s0_, _ = project_to_path(s_xy, R_, path_cum(R_))
                ch = np.hypot(np.diff(s_xy[:, 0]), np.diff(s_xy[:, 1]))
                if (np.diff(s0_) - ch > np.maximum(DETOUR_MAX_M, 0.5 * ch)).any():
                    R_, kind = None, "detour"
                else:
                    kind = "osm"
        ref[p["uid"]] = R_; kinds[kind] += 1
    print(f"matching: {dict(kinds)}")

    out_frames, new_spans, src, next_seg = [], {}, {}, AUG_SEG_START
    for p in passes:
        R_ = ref[p["uid"]]
        if R_ is None:
            continue
        cumR = path_cum(R_); L = cumR[-1]
        if L <= 1.0:
            continue
        xy0, dt0, n = p["xy"], p["dt"], len(p["xy"])
        s0, e0 = project_to_path(xy0, R_, cumR)
        u0 = np.clip(s0 / L, 0.0, 1.0)
        fixed = np.union1d(np.where((s0 <= 1e-6) | (s0 >= L - 1e-6))[0], [0, n - 1])   # endpoints and off-road
        for _ in range(N_VARS):
            pt, nrm = point_at_arc(R_, cumR, L * random_warp(u0))
            e_new = np.clip(np.clip(e0, -E_CLIP_M, E_CLIP_M) + ar1(n, LAT_JITTER_M), -ROAD_SNAP_MAX_M, ROAD_SNAP_MAX_M)
            xy = pt + e_new[:, None] * nrm
            xy[fixed] = xy0[fixed] + rng.normal(0.0, LAT_JITTER_M, (len(fixed), 2))
            x, y = xy[:, 0], xy[:, 1]
            dt = np.clip(dt0 + rng.integers(-DT_JITTER_S, DT_JITTER_S + 1, n), 1.0, None)
            shift = pd.Timedelta(seconds=int(rng.integers(-TIME_SHIFT_MIN * 60, TIME_SHIFT_MIN * 60 + 1)))
            t = p["t0"] + shift + pd.to_timedelta(np.round(np.r_[0.0, np.cumsum(dt[:-1])]), unit="s")
            dx = np.diff(x, append=x[-1]); dx[-1] = 0.0
            dy = np.diff(y, append=y[-1]); dy[-1] = 0.0
            dt_last = dt.copy(); dt_last[-1] = 60.0
            hod = t.hour + t.minute / 60.0 + t.second / 3600.0
            dow = t.dayofweek + hod / 24.0
            xc, yc = x - x_home, y - y_home
            out_frames.append(pd.DataFrame({
                "real_time": t, "dx": dx, "dy": dy, "x": x, "y": y, "B": 1, "seg_id": next_seg, "dt_s": dt_last,
                "is_seg_start": np.r_[1, np.zeros(n - 1, dtype=int)], "is_seg_end": np.r_[np.zeros(n - 1, dtype=int), 1],
                "dt_norm": np.clip((dt_last - 60.0) / 60.0, -1, 1),
                "dx_norm": np.clip(dx / STEP_SCALE, -CLIP_NORM, CLIP_NORM), "dy_norm": np.clip(dy / STEP_SCALE, -CLIP_NORM, CLIP_NORM),
                "tod_sin": np.sin(2 * np.pi * hod / 24.0), "tod_cos": np.cos(2 * np.pi * hod / 24.0),
                "dow_sin": np.sin(2 * np.pi * dow / 7.0), "dow_cos": np.cos(2 * np.pi * dow / 7.0),
                "xc_m": xc, "yc_m": yc, "d_home_m": np.hypot(xc, yc), "xc_norm": xc / SCALE_M, "yc_norm": yc / SCALE_M}))
            new_spans[str(next_seg)] = [[p["ts_loc"], p["te_loc"]]]
            src[str(next_seg)] = p["uid"]
            next_seg += 1

    full = pd.concat([base] + [f_[base.columns] for f_ in out_frames], ignore_index=True)
    full.to_csv("segments_part2.csv", index=False)
    spans_out = {str(k): [list(s) for s in v] for k, v in tspans0.items()} | new_spans
    json.dump(spans_out, open("trip_spans.json", "w"))
    json.dump({"fold": FOLD, "aug_seg_start": AUG_SEG_START, "aug_seg_end": next_seg - 1, "n_vars": N_VARS,
               "time_shift_min": TIME_SHIFT_MIN, "seg_src": src}, open("aug_info.json", "w"))
    print(f"variants: {len(new_spans)} segments from {len(new_spans) // N_VARS} trips | "
          f"points {len(base):,} → {len(full):,}")

## 6. Windows

Windows of `W` points with stride `STRIDE`. The last one is aligned with the end of the segment. Segments with 7–47 points give one padded window, while those with < 7 are ignored.

Indicators per window:
- `is_on_trip`: contains a whole trip with margin `MARGIN`, or lies entirely inside a trip > `MAX_SPAN`,
- `has_transit`:  distance ≥ ≈ 1.3 km,
- `med_dist_home_m`: median distance from home.

In [ ]:
MIN_USABLE = 7

seg_df = pd.read_csv("segments_part2.csv")
rows = []
for seg_id, n in seg_df.groupby("seg_id").size().items():
    if n < MIN_USABLE:
        continue
    if n < W:
        rows.append((seg_id, 0, n))
        continue
    starts = list(range(0, n - W + 1, STRIDE))
    if starts[-1] != n - W:
        starts.append(n - W)
    rows += [(seg_id, s, s + W) for s in starts]
wi = pd.DataFrame(rows, columns=["seg_id", "start", "end"])
wi["n_real"] = wi.end - wi.start

by_seg = {sid: g for sid, g in seg_df.groupby("seg_id")}
d_home, travel = [], []
for r in wi.itertuples():
    ch = by_seg[r.seg_id].iloc[r.start:r.end]
    d_home.append(np.hypot(ch.xc_m, ch.yc_m).median())
    travel.append(np.hypot(ch.dx, ch.dy).sum())
wi["med_dist_home_m"] = d_home
wi["has_transit"] = (np.log1p(travel) >= 7.2).astype(int)

trip_spans = {int(k): [tuple(s) for s in v] for k, v in json.load(open("trip_spans.json")).items()}


def trip_rule(r):
    rule = 0
    for s_, e_ in trip_spans.get(int(r.seg_id), []):
        if r.start <= s_ - MARGIN and r.end >= e_ + MARGIN:
            return 1
        if e_ - s_ > MAX_SPAN and r.start >= s_ and r.end - 1 <= e_:
            rule = 2
    return rule


rule = wi.apply(trip_rule, axis=1)
wi["is_on_trip"] = rule > 0
wi.to_csv("window_index.csv", index=False)

n_seg = seg_df.seg_id.nunique()
print(f"segments: {n_seg:,}, with < {MIN_USABLE} points: {int((seg_df.groupby('seg_id').size() < MIN_USABLE).sum())}")
print(f"windows: {len(wi):,} (with padding: {int((wi.n_real < W).sum())}) | is_on_trip: {int(wi.is_on_trip.sum()):,} "
      f"(rule 1: {int((rule == 1).sum())}, rule 2: {int((rule == 2).sum())}) | has_transit: {wi.has_transit.mean():.0%}")

## 7. Dataset and masking

Each sample: `(x, true_xy, mask_real, mask_hidden)` with 13 features per point.

- **Gap** `a … b−1`: at the masked points `xc, yc, dx, dy = 0` and `is_masked = 1`. The `dx, dy` of `a−1` is also set to zero. `gap_frac` gives the temporal position inside the gap.
- **Gap length** 
- **Gap position**
- **Waypoints**

In [ ]:
from torch.utils.data import Dataset, DataLoader

FEATURE_COLS = ["xc_norm", "yc_norm", "dx_norm", "dy_norm", "tod_sin", "tod_cos", "dow_sin", "dow_cos",
                "dt_norm", "is_seg_start", "is_seg_end", "is_masked", "gap_frac"]
IDX_XC, IDX_YC, IDX_DX, IDX_DY = (FEATURE_COLS.index(c) for c in ["xc_norm", "yc_norm", "dx_norm", "dy_norm"])
IDX_DT_NORM, IDX_IS_MASKED, IDX_GAP_FRAC = (FEATURE_COLS.index(c) for c in ["dt_norm", "is_masked", "gap_frac"])

MIN_MASK_LEN = 2
MASK_SENTINEL = 0.0
TRIP_BIAS, TRANSIT_BIAS = 0.6, 0.7
WAYPOINTS_BY_LEN = ((25, 3), (5, 2), (3, 1))     # cap on waypoints per gap length
WP_SEP_M = 150.0
WAYPOINT_CRITERIA = ("ped", "sed", "sed")


def sample_gap_len(lo, hi):
    """Gap length in [lo, hi] with p(L) ∝ 1/√L."""
    L = np.arange(lo, hi + 1); p = 1.0 / np.sqrt(L)
    return int(np.random.choice(L, p=p / p.sum()))


def worker_init(worker_id):
    """Different numpy seed in each DataLoader worker."""
    np.random.seed((torch.initial_seed() + worker_id) % 2 ** 32)


class TrajectoryWindows(Dataset):

    def __init__(self, seg_df, windows, masking=True, waypoint_mode="key", waypoint_random=True):
        df = seg_df.assign(is_masked=0.0, gap_frac=0.0)
        self.masking, self.waypoint_mode, self.waypoint_random = masking, waypoint_mode, waypoint_random
        self.windows = windows.reset_index(drop=True)
        self.by_seg = {sid: g[FEATURE_COLS].to_numpy(dtype="float32") for sid, g in df.groupby("seg_id")}
        self.by_seg_step = {sid: np.hypot(g.dx.to_numpy(), g.dy.to_numpy()).astype("float32")
                            for sid, g in df.groupby("seg_id")}
        spans = json.load(open("trip_spans.json"))
        self.trip_spans = {int(k): [(int(s), int(e)) for s, e in v if (e - s) + 2 <= W] for k, v in spans.items()}
        self.max_mask_len_now = MIN_MASK_LEN

    def set_curriculum(self, max_len):
        self.max_mask_len_now = max_len

    def __len__(self):
        return len(self.windows)

    @staticmethod
    def _apply_mask(x, mask_hidden, a, b, n_real):
        """Masks positions a..b-1. The endpoints a-1 and b stay visible."""
        a, b = max(int(a), 1), min(int(b), n_real - 1)
        if b - a < 1:
            return False
        x[a:b, [IDX_XC, IDX_YC]] = MASK_SENTINEL
        x[a - 1:b, [IDX_DX, IDX_DY]] = MASK_SENTINEL               # the step a-1 → a is also unknown
        x[a:b, IDX_IS_MASKED] = 1.0
        mask_hidden[a:b] = True
        dt = np.clip(x[a - 1:b, IDX_DT_NORM] * 60.0 + 60.0, 1e-3, None)   # dt_s[i] = t[i+1] - t[i]
        cum = np.cumsum(dt)
        x[a:b, IDX_GAP_FRAC] = cum[:-1] / cum[-1] if cum[-1] > 0 else np.arange(1, b - a + 1) / (b - a + 1)
        return True

    def _mask_gap(self, x, mask_hidden, a, b, n_real):
        """Gap a..b-1 with 0–3 visible waypoints inside it."""
        a, b = max(int(a), 1), min(int(b), n_real - 1)
        ks = []
        if self.waypoint_mode == "key":
            dt = np.clip(x[:n_real, IDX_DT_NORM] * 60.0 + 60.0, 1e-3, None)
            t = np.r_[0.0, np.cumsum(dt[:-1])]                            # time of each point (s)
            P = x[:n_real][:, [IDX_XC, IDX_YC]]
            cap = next((k for thr, k in WAYPOINTS_BY_LEN if b - a >= thr), 0)
            if cap > 1 and self.waypoint_random:
                cap = int(np.random.randint(1, cap + 1))
            sep = WP_SEP_M / SCALE_M
            for crit in WAYPOINT_CRITERIA[:cap]:
                bounds = [a - 1] + sorted(ks) + [b]                       # visible endpoints of the sub-gaps
                cand, score = [], []
                for s_, e_ in zip(bounds[:-1], bounds[1:]):
                    for i in range(s_ + 2, e_ - 1):                       # ≥ 1 masked point on each side
                        if min(np.hypot(*(P[i] - P[v])) for v in bounds) < sep:
                            continue
                        cand.append(i)
                        if crit == "ped":                                 
                            d, r = P[e_] - P[s_], P[i] - P[s_]
                            nd = np.hypot(*d)
                            score.append(abs(d[0] * r[1] - d[1] * r[0]) / nd if nd > 0 else np.hypot(*r))
                        else:                                            
                            f = (t[i] - t[s_]) / max(t[e_] - t[s_], 1e-9)
                            score.append(np.hypot(*(P[i] - P[s_] - f * (P[e_] - P[s_]))))
                if not cand:
                    break
                ks.append(cand[int(np.argmax(score))])
        edges = [a - 1] + sorted(ks) + [b]
        ok = False
        for s_, e_ in zip(edges[:-1], edges[1:]):                         # each sub-gap separately
            ok = self._apply_mask(x, mask_hidden, s_ + 1, e_, n_real) or ok
        return ok

    def __getitem__(self, i):
        r = self.windows.iloc[i]
        seg_id, start, end = int(r.seg_id), int(r.start), int(r.end)
        real = self.by_seg[seg_id][start:end]
        n_real = len(real)
        x = np.zeros((W, len(FEATURE_COLS)), dtype="float32")
        x[:n_real] = real
        true_xy = x[:, [IDX_XC, IDX_YC]].copy()                          # target, before masking
        mask_real = np.zeros(W, dtype=bool); mask_real[:n_real] = True
        mask_hidden = np.zeros(W, dtype=bool)

        if self.masking and n_real - 2 >= MIN_MASK_LEN:
            L = sample_gap_len(MIN_MASK_LEN, min(self.max_mask_len_now, n_real - 2))
            trips = [(ts - start, te - start) for ts, te in self.trip_spans.get(seg_id, [])
                     if ts >= start and te <= end and te - ts >= MIN_MASK_LEN]
            a = None
            if trips and np.random.rand() < TRIP_BIAS:                    # inside a trip
                ts, te = trips[np.random.randint(len(trips))]
                lo, hi = max(1, ts), min(te - L, n_real - L - 1)
                if hi >= lo:
                    a = np.random.randint(lo, hi + 1)
            if a is None:
                if n_real - L - 1 >= 1 and np.random.rand() < TRANSIT_BIAS:   # according to movement
                    cand = np.arange(1, n_real - L)
                    cs = np.r_[0.0, np.cumsum(self.by_seg_step[seg_id][start:end])]
                    w = cs[cand + L] - cs[cand] + 1e-3
                    a = int(np.random.choice(cand, p=w / w.sum()))
                else:
                    a = np.random.randint(1, n_real - L)
            self._mask_gap(x, mask_hidden, a, a + L, n_real)

        return (torch.from_numpy(x), torch.from_numpy(true_xy),
                torch.from_numpy(mask_real), torch.from_numpy(mask_hidden))


print(f"SCALE_M = {SCALE_M:.1f} m | {len(FEATURE_COLS)} features")
_L = np.arange(MIN_MASK_LEN, 32)
for name, p in [("uniform", np.ones(len(_L))), ("1/√L", 1 / np.sqrt(_L))]:
    p = p / p.sum()
    print(f"p(L) in [2, 31], {name}: L ≤ 9 {p[_L <= 9].sum():.0%} | L ≥ 12 {p[_L >= 12].sum():.0%}")

## 8. Train / validation / test and evaluation gaps

- **Test / validation:** each day in non-overlapping windows of `W` points.
- **Train:** sliding windows with all points on train days.
- **Evaluation gaps:** in each window one gap for every L ∈ {3, 6, 11, 20, 31} that fits, at the position with the maximum movement. With movement if this is ≥ 500 m. The name of each gap contains the fold (`f{FOLD}_w{n}_L{L}`), so that it is unique across all folds.
- **Familiarity:** median of the number of reference days with a record in the same or a neighboring 100 m cell.

In [ ]:
seg_df_full = pd.read_csv("segments_part2.csv")
seg_df_full["real_time"] = pd.to_datetime(seg_df_full.real_time)
window_index = pd.read_csv("window_index.csv")

# the augmentation segments (seg_id ≥ AUG_SEG_START) are train only
AUG_SEG_START = (json.load(open("aug_info.json"))["aug_seg_start"] if os.path.exists("aug_info.json")
                 else int(seg_df_full.seg_id.max()) + 1)
MOVING_M = 500.0
FAM_CELL_M = 100.0
FAM_BINS, FAM_LABELS = [-np.inf, 1, 3, 10, np.inf], ["0", "1-2", "3-9", "10+"]   


def role_point_mask(days):
    m = (seg_df_full.seg_id < AUG_SEG_START) & day_str(seg_df_full.real_time).isin(days)
    return {sid: v.to_numpy() for sid, v in m.groupby(seg_df_full.seg_id) if v.any()}


def role_windows(pmask):
    """Non-overlapping windows of W points, on consecutive points of the same day."""
    days_by_seg = {sid: day_str(g.real_time).to_numpy() for sid, g in seg_df_full.groupby("seg_id")}
    rows = []
    for sid, m in pmask.items():
        idx, dd = np.where(m)[0], days_by_seg[sid]
        br = np.where((np.diff(idx) > 1) | (dd[idx[1:]] != dd[idx[:-1]]))[0] + 1
        for run in np.split(idx, br):
            for s0 in range(run[0], run[-1] + 1, W):
                e0 = min(s0 + W, run[-1] + 1)
                if e0 - s0 >= MIN_USABLE:
                    rows.append((sid, s0, e0, dd[s0]))
    return pd.DataFrame(rows, columns=["seg_id", "start", "end", "day"])


def n_points_in(pmask, wins):
    """Number of points of each window that belong to pmask."""
    cs = {sid: np.r_[0, np.cumsum(m)] for sid, m in pmask.items()}
    return np.array([cs[s][e] - cs[s][b] if s in cs else 0 for s, b, e in zip(wins.seg_id, wins.start, wins.end)])


test_mask, val_mask = role_point_mask(TEST_DAYS), role_point_mask(VAL_DAYS)
TEST_WINDOWS, VAL_WINDOWS = role_windows(test_mask), role_windows(val_mask)
n_t, n_v = n_points_in(test_mask, window_index), n_points_in(val_mask, window_index)
train_windows = window_index[(n_t == 0) & (n_v == 0)].reset_index(drop=True)
dev_windows = window_index[n_t == 0].reset_index(drop=True)
n_r = (window_index.end - window_index.start).to_numpy() - n_t - n_v
cross = (n_t > 0).astype(int) + (n_v > 0) + (n_r > 0) > 1               # points from days of different sets
print(f"windows crossing a role boundary (discarded): {int(cross.sum())}")
print(f"FOLD {FOLD}: windows train {len(train_windows):,} | dev {len(dev_windows):,} | "
      f"validation {len(VAL_WINDOWS):,} | test {len(TEST_WINDOWS):,}")


def build_jobs(wins, waypoint_mode="key"):
    """Fixed evaluation gaps: one per window and length L, at the position with the maximum movement."""
    ds = TrajectoryWindows(seg_df_full, wins[["seg_id", "start", "end"]], masking=False,
                           waypoint_mode=waypoint_mode, waypoint_random=False)
    jobs, Xs, Ys, MRs, MHs = [], [], [], [], []
    for i, r in enumerate(wins.itertuples()):
        st = ds.by_seg_step[r.seg_id][r.start:r.end]; n = len(st)
        x0, y0, mr, _ = ds[i]
        for L in GAP_LENGTHS_EVAL:
            if L + 2 > n:
                continue
            mov = np.array([st[a - 1:a + L].sum() for a in range(1, n - L)])   # distance from a-1 to b
            a = 1 + int(mov.argmax())
            x_np, mh = x0.numpy().copy(), np.zeros(W, dtype=bool)
            ds._mask_gap(x_np, mh, a, a + L, n)
            jobs.append(dict(win_num=i, win_id=f"f{FOLD}_w{i}_L{L}", seg_id=r.seg_id, day=r.day,
                             start=r.start, end=r.end, a=a, b=a + L, gap_len=L, move_m=float(mov.max())))
            Xs.append(torch.from_numpy(x_np)); Ys.append(y0); MRs.append(mr); MHs.append(torch.from_numpy(mh))
    return dict(jobs=jobs, X=torch.stack(Xs), Y=torch.stack(Ys), MR=torch.stack(MRs), MH=torch.stack(MHs))


def gap_familiarity(TJ, days):
    """Familiarity of each gap with respect to days (real segments only)."""
    s_ = seg_df_full[seg_df_full.seg_id < AUG_SEG_START]
    dd = day_str(s_.real_time).to_numpy()
    m = np.isin(dd, list(days))
    c = pd.DataFrame({"cx": (s_.x.to_numpy()[m] // FAM_CELL_M).astype(int),
                      "cy": (s_.y.to_numpy()[m] // FAM_CELL_M).astype(int), "d": dd[m]}).drop_duplicates()
    cells = c.groupby(["cx", "cy"]).d.apply(frozenset).to_dict()
    xy = {sid: g[["x", "y"]].to_numpy() for sid, g in seg_df_full.groupby("seg_id")}
    cache = {}

    def n_days(cx, cy):
        if (cx, cy) not in cache:
            cache[(cx, cy)] = len(frozenset().union(*(cells.get((cx + i, cy + k), frozenset())
                                                      for i in (-1, 0, 1) for k in (-1, 0, 1))))
        return cache[(cx, cy)]

    return np.array([float(np.median([n_days(int(x // FAM_CELL_M), int(y // FAM_CELL_M))
                                      for x, y in xy[j["seg_id"]][j["start"] + j["a"]: j["start"] + j["b"]]]))
                     for j in TJ["jobs"]])

## 9. Model and loss function

- **Model:** Conv1d  + Wpos → Transformer encoder (pre-LN).
- **Iterative refinement:** 3 rounds. Each round takes as input the predictions of the previous one (with detach).
- **Loss**: MIT on the masked points +  ORT on the visible points (before replacement) +  step loss, in each round with weights 0.3 / 0.3 / 1.

In [ ]:
import math
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler

REF_M = 100.0
REF = REF_M / SCALE_M
N_REFINE_ROUNDS, ROUND_WEIGHTS = 3, [0.3, 0.3, 1.0]


class TrajectoryTransformer(nn.Module):

    def __init__(self, d_model=D_MODEL, nhead=4, num_layers=N_LAYERS, dim_ff=512, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Conv1d(len(FEATURE_COLS), d_model, kernel_size=3, padding=1)   # sees t-1, t, t+1
        self.pos_embed = nn.Embedding(W, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers, enable_nested_tensor=False)
        self.coord_head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, 2))
        nn.init.zeros_(self.coord_head[-1].weight)          # initial prediction = home
        nn.init.zeros_(self.coord_head[-1].bias)

    def forward(self, x, mask_real, mask_hidden):
        B, Wl, _ = x.shape
        h = self.input_proj(x.transpose(1, 2)).transpose(1, 2)
        pos = torch.arange(Wl, device=x.device).unsqueeze(0).expand(B, Wl)
        raw = self.coord_head(self.encoder(h + self.pos_embed(pos), src_key_padding_mask=~mask_real))
        obs = x[..., [IDX_XC, IDX_YC]]
        return torch.where(mask_hidden.unsqueeze(-1), raw, obs), raw


print(f"model parameters: {sum(p.numel() for p in TrajectoryTransformer().parameters()):,}")


def run_refinement_rounds(model, x, mask_real, mask_hidden, return_raw=False):
    """Each round feeds the predictions (and dx, dy) back as input."""
    nxt_h = torch.zeros_like(mask_hidden); nxt_h[:, :-1] = mask_hidden[:, 1:]
    nxt_r = torch.zeros_like(mask_real); nxt_r[:, :-1] = mask_real[:, 1:]
    dmask = (mask_hidden | nxt_h) & mask_real & nxt_r            # unknown dx[t] = x[t+1] - x[t]
    k = SCALE_M / STEP_SCALE
    x_r, outs, raws = x, [], []
    for r in range(N_REFINE_ROUNDS):
        coords, raw = model(x_r, mask_real, mask_hidden)
        outs.append(coords); raws.append(raw)
        if r < N_REFINE_ROUNDS - 1:
            prev = coords.detach()
            xc = torch.where(mask_hidden, prev[..., 0], x_r[..., IDX_XC])
            yc = torch.where(mask_hidden, prev[..., 1], x_r[..., IDX_YC])
            nx_ = torch.zeros_like(xc); nx_[:, :-1] = xc[:, 1:]
            ny_ = torch.zeros_like(yc); ny_[:, :-1] = yc[:, 1:]
            cols = list(torch.unbind(x_r, dim=2))
            cols[IDX_XC], cols[IDX_YC] = xc, yc
            cols[IDX_DX] = torch.where(dmask, ((nx_ - xc) * k).clamp(-CLIP_NORM, CLIP_NORM), cols[IDX_DX])
            cols[IDX_DY] = torch.where(dmask, ((ny_ - yc) * k).clamp(-CLIP_NORM, CLIP_NORM), cols[IDX_DY])
            x_r = torch.stack(cols, dim=2)
    return (outs, raws) if return_raw else outs


def step_len_loss(coords, true_xy, mask_hidden, mask_real):
    """Difference between predicted and true step length, on pairs with ≥ 1 masked point."""
    dp = torch.linalg.norm(coords[:, 1:] - coords[:, :-1], dim=-1)
    dt = torch.linalg.norm(true_xy[:, 1:] - true_xy[:, :-1], dim=-1)
    m = ((mask_hidden[:, :-1] | mask_hidden[:, 1:]) & mask_real[:, :-1] & mask_real[:, 1:]).float()
    return ((((dp - dt) / REF) ** 2) * m).sum() / m.sum().clamp(min=1)


def compute_loss(coords, raw, true_xy, mask_real, mask_hidden):
    vis = mask_real & ~mask_hidden
    mit = ((((coords - true_xy) / REF) ** 2).sum(-1) * mask_hidden).sum() / mask_hidden.sum().clamp(min=1)
    ort = ((((raw - true_xy) / REF) ** 2).sum(-1) * vis).sum() / vis.sum().clamp(min=1)
    st = step_len_loss(coords, true_xy, mask_hidden, mask_real)
    return mit + LAMBDA_ORT * ort + W_STEP * st, mit.item()

## 10. Training

- **Optimization:** AdamW, batch 64, warmup and cosine decay over 100 epochs, gradient clipping at 1.
- **Curriculum:** maximum gap length 3 → 6 → 10 → 15 → 20 → 31 at epochs 0, 5, 10, 20, 30, 40.
- **Sampler:** 50% windows with a trip, 25% with movement, 15% stays > 1 km from home, 10% the rest.
- **Step 1:** 100 epochs on train and validation at every epoch, on the fixed gaps. Criterion = 0.9 × (gaps with movement) + 0.1 × (without movement).
- **Step 2 (refit)** 

In [ ]:
FINAL_EPOCHS = 100
E_RANGE = (40, FINAL_EPOCHS)
CURRICULUM = [(0, 3), (5, 6), (10, 10), (20, 15), (30, 20), (40, 31)]     # (from epoch, maximum L)
AWAY_M = 1000.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def curriculum_max(epoch):
    return min([v for e, v in CURRICULUM if epoch >= e][-1], W - 4)


def build_sampler(wins, trip=0.50, transit=0.25, away=0.15):
    """Each category gets a fixed share of the samples of each epoch."""
    is_tr = wins.is_on_trip.to_numpy().astype(bool)
    is_t = wins.has_transit.to_numpy().astype(bool) & ~is_tr
    is_a = (wins.med_dist_home_m.to_numpy() > AWAY_M) & ~is_tr & ~is_t
    is_p = ~is_tr & ~is_t & ~is_a
    w = np.zeros(len(wins))
    for m, frac in [(is_tr, trip), (is_t, transit), (is_a, away), (is_p, 1.0 - trip - transit - away)]:
        if m.any():
            w[m] = frac / m.sum()
    print("  sampler (windows): " + ", ".join(f"{n} {int(m.sum()):,}" for n, m in
                                                 [("trip", is_tr), ("movement", is_t), ("far", is_a), ("rest", is_p)]))
    print(f"  trip, movement and far: {(~is_p).mean():.0%} of the windows")
    return WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(wins), replacement=True)


@torch.no_grad()
def eval_jobs(model, TJ):
    """RMSE (m) on the fixed gaps: with / without movement, all and familiar only."""
    model.eval()
    se = []
    for k in range(0, len(TJ["jobs"]), 256):
        mh = TJ["MH"][k:k + 256].to(DEVICE)
        c = run_refinement_rounds(model, TJ["X"][k:k + 256].to(DEVICE), TJ["MR"][k:k + 256].to(DEVICE), mh)[-1]
        se.append(((((c - TJ["Y"][k:k + 256].to(DEVICE)) ** 2).sum(-1) * mh).sum(1)).cpu())
    se = torch.cat(se).double().numpy() * SCALE_M ** 2
    npt = TJ["MH"].sum(1).double().numpy()
    mov = np.array([j["move_m"] >= MOVING_M for j in TJ["jobs"]])
    fam = np.asarray(TJ["fam"]) >= 1
    r = lambda m: float(np.sqrt(se[m].sum() / max(npt[m].sum(), 1)))
    return {"rmse_all": r(np.ones_like(mov)), "rmse_mov1": r(mov), "rmse_mov0": r(~mov),
            "rmse_mov1_fam": r(mov & fam), "rmse_mov0_fam": r(~mov & fam)}


def train(wins, epochs, val_jobs=None):
    """Training from scratch. With val_jobs: evaluation at every epoch."""
    ds = TrajectoryWindows(seg_df_full, wins)
    loader = DataLoader(ds, batch_size=64, sampler=build_sampler(ds.windows), pin_memory=True,
                        num_workers=2, worker_init_fn=worker_init)
    model = TrajectoryTransformer().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    total = FINAL_EPOCHS * len(loader)                      # same LR schedule in step 1 and refit
    warmup = min(200, max(1, total // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup))))
    hist = []
    for epoch in range(epochs):
        ds.set_curriculum(curriculum_max(epoch))
        model.train()
        mits = []
        for xb, yb, mrb, mhb in loader:
            xb, yb, mrb, mhb = xb.to(DEVICE), yb.to(DEVICE), mrb.to(DEVICE), mhb.to(DEVICE)
            outs, raws = run_refinement_rounds(model, xb, mrb, mhb, return_raw=True)
            loss = 0.0
            for w_r, c_r, r_r in zip(ROUND_WEIGHTS, outs, raws):
                l, mit = compute_loss(c_r, r_r, yb, mrb, mhb)
                loss = loss + w_r * l
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step(); sched.step()
            mits.append(mit)                                 # MIT of the last round
        rec = {"epoch": epoch, "coord_loss": float(np.mean(mits))}
        line = f"  epoch {epoch + 1:3d}: train ≈ {np.sqrt(rec['coord_loss']) * REF_M:5.0f} m"
        if val_jobs is not None:
            rec.update(eval_jobs(model, val_jobs))
            line += f" | val with movement (familiar) {rec['rmse_mov1_fam']:5.0f} m"
        hist.append(rec); print(line)
    return model, pd.DataFrame(hist)


# step 1: train + validation → E
VAL_JOBS = build_jobs(VAL_WINDOWS)
VAL_JOBS["fam"] = gap_familiarity(VAL_JOBS, TRAIN_DAYS)          # familiarity with respect to the train days
print(f"\nstep 1: {len(train_windows):,} train windows | {len(VAL_JOBS['jobs']):,} validation gaps")
_, vc = train(train_windows, FINAL_EPOCHS, val_jobs=VAL_JOBS)
vc.insert(0, "n_epochs", vc.epoch + 1)
vc["crit"] = (0.9 * vc.rmse_mov1_fam / vc.rmse_mov1_fam.min() + 0.1 * vc.rmse_mov0_fam / vc.rmse_mov0_fam.min())
vc["crit_ma3"] = vc.crit.rolling(3, center=True).mean()
vc.to_csv(f"val_curve_fold{FOLD}.csv", index=False)
c = vc[vc.n_epochs.between(*E_RANGE)]
E_FOLD = int(c.loc[c.crit_ma3.idxmin(), "n_epochs"])
json.dump({"fold": FOLD, "E": E_FOLD, "w_step": W_STEP}, open(f"E_fold{FOLD}.json", "w"))
print(f"E = {E_FOLD} | val with movement (familiar): {vc.set_index('n_epochs').rmse_mov1_fam[E_FOLD]:.0f} m")

# step 2: refit on the whole dev set for E epochs
print(f"\nstep 2: {len(dev_windows):,} dev windows, {E_FOLD} epochs")
final_model, _ = train(dev_windows, E_FOLD)
torch.save(final_model.state_dict(), f"model_fold{FOLD}_refit.pth")

## 11. Prediction on the test set

Same gaps in two scenarios.

In [ ]:
from pyproj import Transformer

GAP_STEP_FLOOR_M = 10.0
to_ll = Transformer.from_crs("epsg:2100", "epsg:4326", always_xy=True)


def path_dist(px, py, poly):
    """Distance of the point from the polyline poly."""
    a, v = poly[:-1], poly[1:] - poly[:-1]
    t = np.clip(((np.array([px, py]) - a) * v).sum(1) / np.maximum((v ** 2).sum(1), 1e-9), 0.0, 1.0)
    q = a + t[:, None] * v
    return float(np.hypot(px - q[:, 0], py - q[:, 1]).min())


@torch.no_grad()
def predict(model, TJ, scenario):
    """Prediction (last round) on all gaps and a per-point table."""
    model.to(DEVICE).eval()
    preds = np.concatenate([run_refinement_rounds(model, TJ["X"][k:k + 256].to(DEVICE), TJ["MR"][k:k + 256].to(DEVICE),
                                                  TJ["MH"][k:k + 256].to(DEVICE))[-1].cpu().numpy()
                            for k in range(0, len(TJ["jobs"]), 256)])
    by_seg = {sid: g.reset_index(drop=True) for sid, g in seg_df_full.groupby("seg_id")}
    rows = []
    for job, Pp, mh in zip(TJ["jobs"], preds, TJ["MH"].numpy()):
        a, b = job["a"], job["b"]
        g = by_seg[job["seg_id"]].iloc[job["start"]:job["end"]]
        tx, ty, n = g.x.to_numpy(), g.y.to_numpy(), len(g)
        mh = mh[:n]
        px = np.where(mh, x_home + Pp[:n, 0] * SCALE_M, tx)
        py = np.where(mh, y_home + Pp[:n, 1] * SCALE_M, ty)
        poly = np.c_[tx[a - 1:b + 1], ty[a - 1:b + 1]]             # true route of the gap
        lon_p, lat_p = to_ll.transform(px, py); lon_t, lat_t = to_ll.transform(tx, ty)
        st_t = np.hypot(np.diff(tx[a - 1:b + 1]), np.diff(ty[a - 1:b + 1]))
        st_p = np.hypot(np.diff(px[a - 1:b + 1]), np.diff(py[a - 1:b + 1]))
        gstep = max(float(st_t.mean()), GAP_STEP_FLOOR_M)
        cv_t = float(st_t.std() / st_t.mean()) if st_t.mean() > 0 else 0.0
        cv_p = float(st_p.std() / st_p.mean()) if st_p.mean() > 0 else 0.0
        for k in range(n):
            err = float(np.hypot(px[k] - tx[k], py[k] - ty[k])) if mh[k] else 0.0
            rows.append({"fold": FOLD, "day": job["day"], "win_id": job["win_id"], "win_num": job["win_num"],
                         "gap_len": job["gap_len"], "moving": int(job["move_m"] >= MOVING_M),
                         "move_m": round(job["move_m"], 1), "seg_id": job["seg_id"], "step_index": k,
                         "real_time": g.real_time.iloc[k], "lat": lat_p[k], "lon": lon_p[k],
                         "true_lat": lat_t[k], "true_lon": lon_t[k],
                         "point_type": ("Hidden" if mh[k] else "Anchor" if k in (a - 1, b)
                                        else "Waypoint" if a <= k < b else "Visible"),
                         "is_hidden": int(mh[k]), "err_m": err,
                         "path_err_m": path_dist(px[k], py[k], poly) if mh[k] else 0.0,
                         "scenario": scenario, "fam_days": job["fam_days"],
                         "gap_step_m": round(gstep, 1), "err_norm": err / gstep,
                         "step_cv_true": round(cv_t, 3), "step_cv_pred": round(cv_p, 3)})
    d = pd.DataFrame(rows)
    d = d.join(d.groupby("win_num").move_m.max().rename("win_move_m"), on="win_num")
    return d.sort_values(["win_move_m", "win_num", "gap_len", "step_index"],
                         ascending=[False, True, True, True]).reset_index(drop=True)


final_model = TrajectoryTransformer()
final_model.load_state_dict(torch.load(f"model_fold{FOLD}_refit.pth", map_location="cpu"))

TJ = {s: build_jobs(TEST_WINDOWS, waypoint_mode=m) for s, m in [("wp", "key"), ("nowp", "none")]}
assert [(j["seg_id"], j["start"], j["a"], j["b"]) for j in TJ["wp"]["jobs"]] == \
       [(j["seg_id"], j["start"], j["a"], j["b"]) for j in TJ["nowp"]["jobs"]], "different gaps in the two scenarios"
fam = gap_familiarity(TJ["wp"], DEV_DAYS)                     # familiarity with respect to the dev days
preds = {}
for s in TJ:
    for j, f_ in zip(TJ[s]["jobs"], fam):
        j["fam_days"] = f_
    preds[s] = predict(final_model, TJ[s], s)
    preds[s].to_csv(f"pred_fold{FOLD}_{s}.csv", index=False)
    print(f"pred_fold{FOLD}_{s}.csv: {preds[s].win_id.nunique():,} gaps, {int(preds[s].is_hidden.sum()):,} masked points")

## 12. Fold summary

Position error on the masked points of the gaps with movement, per gap length.

In [ ]:
def gap_stats(h):
    return pd.Series({"gaps": h.win_id.nunique(), "RMSE": np.sqrt((h.err_m ** 2).mean()),
                      "median": h.err_m.median(), "p90": h.err_m.quantile(0.9)}).round(0)


for s, d in preds.items():
    h = d[d.is_hidden == 1]
    hm = h[h.moving == 1]
    t = hm.groupby("gap_len").apply(gap_stats, include_groups=False)
    t.loc["all"] = gap_stats(hm)
    print(f"\n{s}: gaps with movement (m)\n{t.to_string()}")
    print(f"gaps without movement: {gap_stats(h[h.moving == 0]).to_dict()}")
    print("per familiarity (gaps with movement): " + str(hm.groupby(pd.cut(hm.fam_days, FAM_BINS, labels=FAM_LABELS, right=False),
                                                              observed=True).apply(gap_stats, include_groups=False).RMSE.to_dict()))